# Legacy final-winners 96-language validity check

This notebook validates the final 96-language winner runs in the same spirit as `rank_expert_sweep_validity_check.ipynb`, but it is artifact-based because the 96-language profile currently has no completed benchmark report.

Validated profile:
`moe_study_final_winners_noctua2_a100_4gpu_96_langs_34p5h`

Validity criteria:
- the planned run set resolves to the three final-winner approaches
- all runs use the same 96-language tokenized dataset rooted at the profile output directory
- tokenized data has a passing `_check.json`
- each run has `trainer_state.json` and `train_results.json`
- each selected adapter checkpoint exists, has `adapter_model.safetensors`, has `adapter_config.json`, has `.adapter_direct_save_complete`, and its checkpoint step matches the final trainer `global_step`
- wall-clock runtime is close to the intended 34.5h budget and ended through the signal-triggered adapter checkpoint path
- actual training exposure does not exceed one epoch
- benchmark and final-eval result coverage are checked explicitly and reported separately from checkpoint validity


In [ ]:
from pathlib import Path
import json
import re
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "outputs").exists():
    ROOT = Path("/scratch/hpc-prf-merlin/joel/moe-study")

PROFILE_FILE = ROOT / "configs/pipeline/profiles/moe_study_rank_expert_sweep_configs/moe_study_final_winners_96_200_noctua2_4gpu.generated.json"
PROFILE_FILES = sorted((ROOT / "configs/pipeline/profiles").rglob("*.json"))
PROFILE_NAME = "moe_study_final_winners_noctua2_a100_4gpu_96_langs_34p5h"
OUTPUT_ROOT = ROOT / "outputs/pipeline_runs/moe_study_final_winners_noctua2_a100_4gpu_96_langs_34p5h"
EXPECTED_RUNS = ["moelora_tr64_e32_k4", "hydralora_r2_heads32", "mola_tr64_e32_k4"]
FULL_TARGETS = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
ATTENTION_TARGETS = {"q_proj", "k_proj", "v_proj", "o_proj"}

print("profile file:", PROFILE_FILE, PROFILE_FILE.exists())
print("profile json files:", len(PROFILE_FILES))
print("output root:", OUTPUT_ROOT, OUTPUT_ROOT.exists())


In [ ]:
def read_json(path):
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return {}


def deep_merge(base, override):
    out = dict(base)
    for key, value in override.items():
        if isinstance(value, dict) and isinstance(out.get(key), dict):
            out[key] = deep_merge(out[key], value)
        else:
            out[key] = value
    return out


def load_all_profiles():
    profiles = {}
    for path in PROFILE_FILES:
        payload = read_json(path)
        if isinstance(payload, dict):
            profiles.update(payload)
    return profiles


ALL_PROFILES = load_all_profiles()


def resolve_profile(profile_name):
    profile = ALL_PROFILES[profile_name]
    parent = profile.get("extends")
    if parent:
        profile = deep_merge(resolve_profile(parent), profile)
    return profile


def norm_targets(value):
    if value is None:
        return []
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        if value.startswith("["):
            try:
                return norm_targets(json.loads(value))
            except Exception:
                pass
        return sorted({part.strip() for part in value.split(",") if part.strip()})
    if isinstance(value, (list, tuple, set)):
        return sorted({str(part).strip() for part in value if str(part).strip()})
    return []


def checkpoint_step(path):
    match = re.search(r"checkpoint-(\d+)_adapter", Path(path).name)
    return int(match.group(1)) if match else None


def latest_adapter_dir(run_dir):
    adapters = sorted(
        Path(run_dir).glob("checkpoint-*_adapter"),
        key=lambda p: checkpoint_step(p) if checkpoint_step(p) is not None else -1,
    )
    return adapters[-1] if adapters else None


def target_policy(kind, targets):
    target_set = set(targets)
    if not targets:
        return "unknown", "target modules unavailable"
    if target_set == FULL_TARGETS:
        return "full", "full q/k/v/o + gate/up/down target set"
    if target_set == ATTENTION_TARGETS:
        return "attention_only", "intended final-winner attention target set"
    return "other", "non-standard target set"


## Planned run set and profile configuration

In [ ]:
profile = resolve_profile(PROFILE_NAME)
runs_cfg = profile.get("training", {}).get("runs", [])
planned = pd.DataFrame([
    {
        "run_id": row.get("id"),
        "kind": row.get("kind"),
        "rank": row.get("lora", {}).get("rank"),
        "alpha": row.get("lora", {}).get("alpha"),
        "target": row.get("lora", {}).get("target", profile.get("training", {}).get("lora", {}).get("target")),
        "batch_size": row.get("train", {}).get("batch_size"),
        "grad_accum": row.get("train", {}).get("grad_accum"),
        "disable_gradient_checkpointing": row.get("train", {}).get("disable_gradient_checkpointing"),
        "fsdp_config": row.get("distributed", profile.get("training", {}).get("distributed", {})).get("config_file"),
    }
    for row in runs_cfg
])

planned_run_set_ok = set(planned.run_id) == set(EXPECTED_RUNS)
display(planned)
display(Markdown(f"**Planned run-set verdict:** expected final winners present exactly = `{planned_run_set_ok}`."))


## Tokenized data check

In [ ]:
tokenized_dir = OUTPUT_ROOT / "tokenized"
check = read_json(tokenized_dir / "_check.json")
language_map = read_json(OUTPUT_ROOT / "language_map_from_csv.json")

train_tokens = sum(check.get("tokens_per_language_by_split", {}).get("train", {}).values())
validation_tokens = sum(check.get("tokens_per_language_by_split", {}).get("validation", {}).values())

data_summary = pd.DataFrame([{
    "tokenized_dir": str(tokenized_dir.relative_to(ROOT)),
    "check_exists": bool(check),
    "check_pass": check.get("pass"),
    "num_languages_check": check.get("num_languages"),
    "num_unique_language_ids": check.get("num_unique_language_ids"),
    "language_map_entries": len(language_map),
    "train_rows": check.get("splits", {}).get("train"),
    "validation_rows": check.get("splits", {}).get("validation"),
    "train_tokens": train_tokens,
    "validation_tokens": validation_tokens,
}])

display(data_summary)

data_ok = bool(check) and check.get("pass") is True and tokenized_dir.exists()
display(Markdown(f"**Tokenized-data verdict:** tokenized data exists and `_check.json` passes = `{data_ok}`. The 96-language CSV plus extra English/script variants resolve to `{check.get('num_languages')}` language/script IDs."))


## Checkpoint, runtime, and exposure validation

In [ ]:
rows = []
for cfg in runs_cfg:
    run_id = cfg["id"]
    kind = cfg["kind"]
    rd = OUTPUT_ROOT / "runs" / run_id
    adapter = latest_adapter_dir(rd)
    trainer_state = read_json(rd / "trainer_state.json")
    train_results = read_json(rd / "train_results.json") or read_json(rd / "all_results.json")
    adapter_cfg = read_json(adapter / "adapter_config.json") if adapter else {}
    targets = norm_targets(adapter_cfg.get("target_modules")) or norm_targets(cfg.get("lora", {}).get("target"))
    policy, reason = target_policy(kind, targets)
    global_step = trainer_state.get("global_step")
    step = checkpoint_step(adapter) if adapter else None
    model_file = adapter / "adapter_model.safetensors" if adapter else None
    marker = adapter / ".adapter_direct_save_complete" if adapter else None
    runtime = train_results.get("train_runtime") or trainer_state.get("train_runtime")
    epoch = train_results.get("epoch", trainer_state.get("epoch"))
    notes = []
    if not rd.exists(): notes.append("missing run dir")
    if not trainer_state: notes.append("missing trainer_state.json")
    if not train_results: notes.append("missing train_results/all_results")
    if not adapter: notes.append("missing adapter checkpoint")
    if adapter and not model_file.exists(): notes.append("missing adapter_model.safetensors")
    if adapter and not adapter_cfg: notes.append("missing adapter_config.json")
    if adapter and not marker.exists(): notes.append("missing direct-save marker")
    if step != global_step: notes.append("checkpoint step != trainer global_step")
    if epoch is not None and float(epoch) > 1.0: notes.append("epoch > 1")
    rows.append({
        "run_id": run_id,
        "kind": kind,
        "run_dir_exists": rd.exists(),
        "trainer_state_present": bool(trainer_state),
        "train_results_present": bool(train_results),
        "global_step": global_step,
        "max_steps": trainer_state.get("max_steps"),
        "epoch": epoch,
        "train_runtime_h": round(float(runtime) / 3600, 3) if runtime else None,
        "train_loss": train_results.get("train_loss"),
        "adapter_dir": adapter.name if adapter else "",
        "checkpoint_step": step,
        "adapter_model_exists": bool(model_file and model_file.exists()),
        "adapter_model_size_mb": round(model_file.stat().st_size / 1_000_000, 1) if model_file and model_file.exists() else None,
        "adapter_config_exists": bool(adapter_cfg),
        "direct_save_marker_exists": bool(marker and marker.exists()),
        "target_modules": ",".join(targets),
        "target_module_policy": policy,
        "target_module_reason": reason,
        "checkpoint_valid": not notes,
        "validity_notes": "; ".join(notes),
    })

run_audit = pd.DataFrame(rows)
display(run_audit)

all_checkpoints_valid = run_audit.checkpoint_valid.all()
all_under_one_epoch = (run_audit.epoch.astype(float) <= 1.0).all()
runtime_close = run_audit.train_runtime_h.between(33.5, 34.75).all()
display(Markdown(f"**Checkpoint/runtime verdict:** checkpoint artifacts valid = `{all_checkpoints_valid}`; no repeated-data exposure = `{all_under_one_epoch}`; runtimes within 33.5-34.75h = `{runtime_close}`."))


## Signal-save evidence from logs

In [ ]:
log_rows = []
for run_id in EXPECTED_RUNS:
    slurm_logs = sorted((OUTPUT_ROOT / "logs/chain/slurm").glob(f"chain_train.{run_id}.*.slurm.log"))
    candidates = []
    for path in slurm_logs:
        text = path.read_text(errors="replace")
        candidates.append((
            "Adapter checkpoint saved at:" in text,
            "[SLURM_END]" in text and "exit_code=0" in text,
            path,
            text,
        ))
    selected = [item for item in candidates if item[0] and item[1]] or candidates[-1:]
    _, _, selected_path, selected_text = selected[-1] if selected else (False, False, None, "")
    job_ids = sorted({a or b for a, b in re.findall(r"job=(\d+)|job_id=(\d+)", selected_text)})
    log_rows.append({
        "run_id": run_id,
        "slurm_log_count": len(slurm_logs),
        "selected_log": selected_path.name if selected_path else "",
        "job_ids_in_selected_log": ",".join(job_ids),
        "saw_usr1": "signal=USR1" in selected_text or "Received SIGUSR1" in selected_text,
        "saw_adapter_save": "Adapter checkpoint saved at:" in selected_text,
        "saw_training_completed": "Training completed" in selected_text,
        "saw_slurm_exit_zero": "[SLURM_END]" in selected_text and "exit_code=0" in selected_text,
        "saw_failure_text_in_selected_log": bool(re.search(r"Traceback| FAILED|RuntimeError|CUDA out of memory", selected_text)),
    })

log_audit = pd.DataFrame(log_rows)
display(log_audit)

log_signal_ok = log_audit[["saw_usr1", "saw_adapter_save", "saw_training_completed", "saw_slurm_exit_zero"]].all(axis=None)
selected_logs_clean = not log_audit.saw_failure_text_in_selected_log.any()
display(Markdown(f"**Signal-save log verdict:** all selected final logs show USR1, adapter save, trainer completion, and SLURM exit 0 = `{log_signal_ok}`; selected final logs contain failure text = `{not selected_logs_clean}`."))


## Benchmark and final-eval coverage

In [ ]:
benchmark_dir = OUTPUT_ROOT / "benchmarks"
final_eval_dir = OUTPUT_ROOT / "final_eval_loss"
summary_csv = OUTPUT_ROOT / "summary.csv"
report_xlsx = OUTPUT_ROOT / f"{PROFILE_NAME}.xlsx"

benchmark_summary = read_json(benchmark_dir / "benchmark_summary.json")
benchmark_check = read_json(benchmark_dir / "_check.json")
benchmark_files = sorted(benchmark_dir.glob("**/*.json")) if benchmark_dir.exists() else []
final_eval_files = sorted(final_eval_dir.glob("**/eval_results.json")) if final_eval_dir.exists() else []
run_local_final_eval_files = sorted((OUTPUT_ROOT / "runs").glob("*/final_eval_loss/eval_results.json"))

runs_in_summary = sorted(str(row.get("run_id")) for row in benchmark_summary.get("runs", []))
run_status_rows = []
for run_id in EXPECTED_RUNS:
    check_row = benchmark_check.get("runs", {}).get(run_id, {})
    run_files = sorted((benchmark_dir / run_id).glob("**/*.json")) if (benchmark_dir / run_id).exists() else []
    run_status_rows.append({
        "run_id": run_id,
        "in_benchmark_summary": run_id in runs_in_summary,
        "benchmark_json_files": len(run_files),
        "expected_tasks": check_row.get("expected"),
        "seen_tasks": check_row.get("seen"),
        "missing_count": check_row.get("missing_count"),
        "check_pass": check_row.get("pass", False),
    })
benchmark_run_coverage = pd.DataFrame(run_status_rows)

display(benchmark_run_coverage)

coverage = pd.DataFrame([{
    "benchmark_dir_exists": benchmark_dir.exists(),
    "benchmark_summary_exists": bool(benchmark_summary),
    "benchmark_check_exists": bool(benchmark_check),
    "benchmark_json_files_total": len(benchmark_files),
    "benchmark_runs_complete": int(benchmark_run_coverage.check_pass.fillna(False).sum()),
    "benchmark_runs_expected": len(EXPECTED_RUNS),
    "final_eval_dir_exists": final_eval_dir.exists(),
    "final_eval_json_files": len(final_eval_files),
    "run_local_final_eval_json_files": len(run_local_final_eval_files),
    "summary_csv_exists": summary_csv.exists(),
    "report_xlsx_exists": report_xlsx.exists(),
}])

display(coverage)

benchmark_complete = benchmark_run_coverage.check_pass.fillna(False).all()
final_eval_complete = len(final_eval_files) + len(run_local_final_eval_files) >= len(EXPECTED_RUNS)
display(Markdown(f"**Benchmark/final-eval verdict:** benchmark results complete for all expected runs = `{benchmark_complete}`; final eval results complete for all expected runs = `{final_eval_complete}`."))


## Overall conclusion

In [ ]:
lines = [
    "## Validity conclusion",
    "",
    f"- Planned final-winner runs present exactly: `{planned_run_set_ok}` ({', '.join(EXPECTED_RUNS)}).",
    f"- Tokenized data exists and passes audit: `{data_ok}`; language/script IDs in `_check.json`: `{check.get('num_languages')}`.",
    f"- Adapter checkpoints are valid for all runs: `{all_checkpoints_valid}`.",
    f"- No run exceeds one epoch: `{all_under_one_epoch}`.",
    f"- Runtime matches the intended 34.5h compute budget: `{runtime_close}`.",
    f"- Log evidence shows signal-triggered final adapter save for all runs: `{log_signal_ok}`.",
    f"- Benchmark results already present: `{benchmark_complete}`.",
    f"- Final eval results already present: `{final_eval_complete}`.",
    "",
    "Overall: the 96-language training runs are valid from the checkpoint/training-artifact side. Benchmark and final-eval stages still need to be run or rerun before benchmark-based conclusions can be drawn from the 96-language checkpoints.",
]
display(Markdown("\n".join(lines)))
